# 20b — CNN on CIFAR-10

Corresponds to: Note 20 — CNN Architecture: Convolutional, Pooling, and Fully Connected Layers

**Libraries introduced in this notebook:** `torchvision.datasets.CIFAR10` — provides the CIFAR-10 dataset (50,000 training images, 32×32 RGB, 10 classes).

**What this notebook demonstrates:**
1. Loading CIFAR-10 and inspecting the data
2. Building a CNN with the layer types from Note 20 — conv, ReLU, pool, flatten, FC
3. Training and tracking loss and accuracy
4. Evaluating per-class accuracy — which classes are hardest
5. Comparing the CNN against a feedforward baseline on the same data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:      {device}')
print(f'PyTorch:     {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')

## Part 1: Load CIFAR-10

50,000 training and 10,000 test images. 32×32 RGB, 10 classes: plane, car, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std= [0.2470, 0.2435, 0.2616]),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=0)

CLASSES = train_set.classes
print(f'Training samples: {len(train_set)}')
print(f'Test samples:     {len(test_set)}')
print(f'Image shape:      {train_set[0][0].shape}  (C x H x W)')
print(f'Classes:          {CLASSES}')

## Part 2: Build the CNN

Two conv blocks (conv + ReLU + maxpool), then flatten and two FC layers — the same structural pattern as LeNet from Note 20, scaled for 32×32 RGB input.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),   # 32x32 -> 16x16
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),   # 16x16 -> 8x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))

model = CNN().to(device)

conv1 = (3*3*3  + 1) * 32
conv2 = (3*3*32 + 1) * 64
fc1   = 64*8*8 * 256 + 256
fc2   = 256 * 10 + 10
print(f'Conv1: (3x3x3+1)x32   = {conv1}')
print(f'Conv2: (3x3x32+1)x64  = {conv2}')
print(f'FC1:   4096x256+256   = {fc1}')
print(f'FC2:   256x10+10      = {fc2}')
print(f'Total: {conv1+conv2+fc1+fc2:,}')

## Part 3: Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader):
    model.train()
    tl, tc, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        tl += loss.item()*len(y); tc += (out.argmax(1)==y).sum().item(); n += len(y)
    return tl/n, tc/n

def eval_epoch(model, loader):
    model.eval()
    tl, tc, n = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            tl += criterion(out,y).item()*len(y); tc += (out.argmax(1)==y).sum().item(); n += len(y)
    return tl/n, tc/n

print(f'{"Epoch":<8} {"Train Loss":<14} {"Train Acc":<14} {"Test Loss":<14} {"Test Acc"}')
print('-' * 60)
for epoch in range(1, 16):
    tr_loss, tr_acc = train_epoch(model, train_loader)
    te_loss, te_acc = eval_epoch(model, test_loader)
    print(f'{epoch:<8} {tr_loss:<14.4f} {tr_acc:<14.4f} {te_loss:<14.4f} {te_acc:.4f}')

## Part 4: Per-Class Accuracy

In [ ]:
model.eval()
class_correct = [0]*10
class_total   = [0]*10
with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        preds = model(X).argmax(1)
        for c in range(10):
            mask = (y == c)
            class_correct[c] += (preds[mask]==c).sum().item()
            class_total[c]   += mask.sum().item()

print(f'{"Class":<12} {"Correct":<10} {"Total":<10} {"Accuracy"}')
print('-' * 42)
for c in range(10):
    print(f'{CLASSES[c]:<12} {class_correct[c]:<10} {class_total[c]:<10} {class_correct[c]/class_total[c]:.4f}')

## Part 5: CNN vs Feedforward Baseline

A feedforward network flattens the 32×32×3 image to 3072 values and has no spatial structure.

In [ ]:
ff_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3072, 512), nn.ReLU(),
    nn.Linear(512,  256), nn.ReLU(),
    nn.Linear(256,   10),
).to(device)
ff_opt = optim.Adam(ff_model.parameters(), lr=1e-3)

print(f'CNN parameters:         {sum(p.numel() for p in model.parameters()):,}')
print(f'Feedforward parameters: {sum(p.numel() for p in ff_model.parameters()):,}')
print()
print('Training feedforward baseline (15 epochs)...')
print(f'{"Epoch":<8} {"Train Loss":<14} {"Train Acc":<14} {"Test Loss":<14} {"Test Acc"}')
print('-' * 60)

for epoch in range(1, 16):
    ff_model.train()
    tl, tc, tn = 0.0, 0, 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        ff_opt.zero_grad()
        out = ff_model(X)
        loss = criterion(out, y)
        loss.backward()
        ff_opt.step()
        tl += loss.item()*len(y); tc += (out.argmax(1)==y).sum().item(); tn += len(y)
    ff_model.eval()
    with torch.no_grad():
        vl, vc, vn = 0.0, 0, 0
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            out = ff_model(X)
            vl += criterion(out,y).item()*len(y); vc += (out.argmax(1)==y).sum().item(); vn += len(y)
    print(f'{epoch:<8} {tl/tn:<14.4f} {tc/tn:<14.4f} {vl/vn:<14.4f} {vc/vn:.4f}')

## Summary

What this notebook demonstrated:

1. **CIFAR-10** — 50,000 training images, 32×32 RGB, 10 classes. Normalised with CIFAR-10 per-channel statistics.
2. **CNN architecture** — two conv blocks (conv + ReLU + maxpool) followed by flatten and two FC layers. Parameter count verified against the Note 20 formula.
3. **Per-class accuracy** — visually similar classes (cat/dog, automobile/truck) have lower accuracy.
4. **CNN vs feedforward** — the CNN achieves significantly higher accuracy than the feedforward baseline because it exploits spatial structure that the feedforward network ignores.